# Práctica de Análisis: Agenda Setting y Producción de Sentido

Como hemos visto a lo largo de los laboratorios, extraer datos no es una meta por sí misma, sino el punto de partida del Análisis Crítico. Hoy aplicaremos este concepto sobre la principal noticia del día.

En el Lab 004 generamos un extracto directo de la portada del diario nacional La Nación (`lanacion_portada.json`). En la presente actividad demostraremos las redes tangibles en las que se enmarca la realidad política e informativa de nuestra sociedad actual; es decir, **analizaremos la Agenda Setting y los focos de atención del medio periodístico**.

Al finalizar este cuaderno de ejercicios, empaquetaremos estos hallazgos analíticos utilizando lo aprendido en el Lab 000: diseñaremos un *Dashboard* Interactivo con la librería **Gradio**.

## Consigna 1: Carga y Exploración del Corpus

A partir del archivo JSON suministrado (`lanacion_portada.json`), lea el contenido empleando la librería principal de manejo estadístico `pandas`. Inspeccione el encuadre estructural (forma y primeros 5 renglones) a fines de asegurar la fidelidad del traspaso de registros.

In [ ]:
import json
import pandas as pd

with open("lanacion_portada.json", "r", encoding="utf-8") as archivo:
    noticias = json.load(archivo)

df = pd.DataFrame(noticias)
print("Forma del dataset:", df.shape)
df.head()


## Consigna 2: Visualización Temática de Agenda Periodística

Con nuestra tabla constituida, buscaremos responder una pregunta clave de investigación: **¿Qué temática decidió priorizar jerárquicamente la línea editorial en su portada de hoy?**

Filtre valores nulos y estructure una representación categórica (ejemplo `sns.barplot()` con orientación horizontal) de modo que exponga de manera irrefutable qué secciones ostentan la mayor cantidad de aparatos editoriales (notas emitidas). Agregue un título informativo y recuerde aplicar principios rigurosos de Data-Ink Ratio.

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

secciones = (
    df["categoria"]
    .dropna()
    .str.strip()
    .value_counts()
    .sort_values(ascending=True)
)

plt.figure(figsize=(10, 6))
sns.barplot(x=secciones.values, y=secciones.index, palette="viridis")
plt.title("Frecuencia de noticias por sección en la portada")
plt.xlabel("Cantidad de notas")
plt.ylabel("Sección")
plt.tight_layout()
plt.show()


## Consigna 3: Pesos del Lenguaje (Redes Léxicas)

Con la temática confirmada, deconstruiremos semánticamente esos focos de atención aislando el lenguaje que envuelve al contenido principal. Emplearemos el *Lollipop Chart* abordado en metodologías previas para proyectar las 15 palabras clave con mayor iteración masiva a lo largo de *toda la portada* titular.

Identifique las iteraciones que se formulan más allá de conectores y artículos nulos (*stop words*). ¿Cuál de estos nodos terminológicos monopoliza el marco informativo de la presente fecha?

In [ ]:
# EJERCICIO: Extraer el Top 15 de palabras más utilizadas en los títulos (ignorando stop words) y representarlas en un Lollipop Chart.
stopwords_es = {
    "de", "la", "que", "el", "en", "y", "a", "los", "del", "se", "las", "por", "un", "para", 
    "con", "no", "una", "su", "al", "lo", "como", "más", "pero", "sus", "le", "ya", "o", "este", 
    "sí", "porque", "esta", "entre", "cuando", "muy", "sin", "sobre", "también", "me", "hasta", 
    "hay", "donde", "quien", "desde", "todo", "nos", "durante", "todos", "uno", "les", "ni", 
    "contra", "otros", "ese", "eso", "ante", "ellos", "e", "esto", "mí", "antes", "algunos", 
    "qué", "unos", "yo", "otro", "otras", "otra", "él", "tanto", "esa", "estos", "mucho", "quienes", 
    "nada", "muchos", "cual", "poco", "ella", "estar", "estas", "algunas", "algo", "nosotros", "mi", 
    "mis", "tú", "te", "ti", "tu", "tus", "ellas", "nosotras", "vosotros", "vosotras", "os", "mío", 
    "mía", "míos", "mías", "tuyo", "tuya", "tuyos", "tuyas", "suyo", "suya", "suyos", "suyas", "nuestro", 
    "nuestra", "nuestros", "nuestras", "vuestro", "vuestra", "vuestros", "vuestras", "esos", "esas", 
    "estoy", "estás", "está", "estamos", "estáis", "están", "esté", "estés", "estemos", "estéis", "estén", "estaré", 
    "estará", "estaremos", "estarán", "estaría", "estarías", "estaríamos", "estarían", "era", "eras", "éramos", 
    "eran", "fui", "fue", "fuimos", "fueron", "ser", "soy", "somos", "son"
}

import re
from collections import Counter

titulos = df["titulo"].dropna().astype(str).str.lower()
texto_titulos = " \n".join(titulos)
tokens = re.findall(r"[a-záéíóúñü]+", texto_titulos)
tokens_filtrados = [token for token in tokens if token not in stopwords_es and len(token) > 2]

top_15 = Counter(tokens_filtrados).most_common(15)
palabras, frecuencias = zip(*top_15)

plt.figure(figsize=(11, 6))
plt.hlines(y=palabras, xmin=0, xmax=frecuencias, color="gray", alpha=0.7)
plt.plot(frecuencias, palabras, "o", color="firebrick")
plt.title("Top 15 palabras más frecuentes en titulares")
plt.xlabel("Frecuencia")
plt.ylabel("Palabra")
plt.tight_layout()
plt.show()


## Consigna Final: Dashboard Dinámico de Sentido con Gradio

Integrar y escalar el diseño. Acaba de confirmar metodológicamente la agenda principal y el vocabulario dominador del periódico entero. Ahora envuélvalo funcionalmente convirtiéndolo en una Interface Web que logre desglosar el contenido respondiendo filtros particulares del investigador.

Deberá programar una Interfaz Gradio (`gr.Interface` o `gr.Blocks`) que exponga un **Desplegable (Dropdown)** poblado con las Secciones halladas (`Política`, `Economía`, `Sociedad`, etc.). 
La selección del usuario invocará una función que:
1. Cribe al DataFrame dejando solamente artículos pertenecientes a dicha sección estival.
2. Genere dos *outputs* (Salidas): 
    * Una tabla (`gr.Dataframe` o texto) enumerando puros títulos resultantes de esa sección selecta.
    * El gráfico del analizador de pesos (`Lollipop de frecuencias`) re-computado exclusivamente sobre el vocabulario de la sección elegida.

> Al correr la Interfaz, notará instantáneamente cómo el eje material y discursivo muta violentamente de polaridad con el simple acto interactivo de mover de 'Política' a 'Cultura'.

In [ ]:
import gradio as gr

def analizar_seccion(seccion):
    filtrado = df[df["categoria"].fillna("").str.strip() == seccion].copy()
    tabla_titulos = filtrado[["titulo"]].dropna().rename(columns={"titulo": "Título"})

    titulos = tabla_titulos["Título"].astype(str).str.lower()
    tokens = re.findall(r"[a-záéíóúñü]+", " \n".join(titulos))
    tokens_filtrados = [t for t in tokens if t not in stopwords_es and len(t) > 2]
    top = Counter(tokens_filtrados).most_common(15)

    fig, ax = plt.subplots(figsize=(10, 5))
    if top:
        palabras, frecs = zip(*top)
        ax.hlines(y=palabras, xmin=0, xmax=frecs, color="gray", alpha=0.7)
        ax.plot(frecs, palabras, "o", color="darkgreen")
        ax.set_title(f"Palabras dominantes en {seccion}")
        ax.set_xlabel("Frecuencia")
        ax.set_ylabel("Palabra")
    else:
        ax.text(0.5, 0.5, "Sin datos léxicos para esta sección", ha="center", va="center")
        ax.set_axis_off()

    return tabla_titulos, fig

secciones_disponibles = sorted(df["categoria"].dropna().str.strip().unique().tolist())

demo = gr.Interface(
    fn=analizar_seccion,
    inputs=gr.Dropdown(choices=secciones_disponibles, label="Sección", value=secciones_disponibles[0] if secciones_disponibles else None),
    outputs=[
        gr.Dataframe(label="Títulos de la sección"),
        gr.Plot(label="Lollipop de frecuencias"),
    ],
    title="Dashboard de Agenda de Medios",
    description="Seleccioná una sección para ver sus titulares y su vocabulario dominante.",
)

# demo.launch()
